In [1]:
#%pip install kaleido
#%pip install mplcursors
#%pip install matplotlib widget

In [2]:
# all libraries used in this notebook
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' #only critical errors
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    LayerNormalization, Input, Conv1D, MaxPooling1D, Flatten,
    Concatenate, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')



In [3]:
df = pd.read_csv("NVDA.csv")

In [4]:
df.head(5)

,Date,Adj Close,Close,High,Low,Open,Volume
0,1999-01-22,0.037615,0.041016,0.048828,0.038802,0.043750,2714688000
1,1999-01-25,0.041556,0.045313,0.045833,0.041016,0.044271,510480000
2,1999-01-26,0.038331,0.041797,0.046745,0.041146,0.045833,343200000
3,1999-01-27,0.038212,0.041667,0.042969,0.039583,0.041927,244368000
4,1999-01-28,0.038092,0.041536,0.041927,0.041276,0.041667,227520000


In [5]:
df = df.sort_values('Date').reset_index(drop=True)

print("Shape:", df.shape)
print(df.head())

Shape: (6558, 7)
         Date  Adj Close     Close      High       Low      Open      Volume
0  1999-01-22   0.037615  0.041016  0.048828  0.038802  0.043750  2714688000
1  1999-01-25   0.041556  0.045313  0.045833  0.041016  0.044271   510480000
2  1999-01-26   0.038331  0.041797  0.046745  0.041146  0.045833   343200000
3  1999-01-27   0.038212  0.041667  0.042969  0.039583  0.041927   244368000
4  1999-01-28   0.038092  0.041536  0.041927  0.041276  0.041667   227520000


### Null Analysis


In [6]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({'Nulls': nulls, 'Porcentage (%)': null_pct})
print(null_report)

           Nulls  Porcentage (%)
Date           0             0.0
Adj Close      0             0.0
Close          0             0.0
High           0             0.0
Low            0             0.0
Open           0             0.0
Volume         0             0.0


In [7]:
dupes = df.duplicated().sum()
print(f"\nDuplicates: {dupes}")
date_dupes = df['Date'].duplicated().sum()
print(f"Duplicated dates: {date_dupes}")


Duplicates: 0
Duplicated dates: 0


In [8]:
df.describe()

,Adj Close,Close,High,Low,Open,Volume
count,6558.000000,6558.000000,6558.000000,6558.000000,6558.000000,6.558000e+03
mean,8.768532,8.795447,8.956567,8.618315,8.795850,5.991103e+08
std,23.907205,23.904882,24.349618,23.419200,23.922708,4.307236e+08
min,0.031286,0.034115,0.035547,0.033333,0.034896,1.968000e+07
25%,0.257739,0.281042,0.288511,0.273354,0.280810,3.384780e+08
50%,0.437176,0.466083,0.472875,0.459250,0.466584,5.002635e+08
75%,4.597059,4.644625,4.724000,4.588750,4.632437,7.307002e+08
max,149.429993,149.429993,153.130005,147.820007,153.029999,9.230856e+09


In [9]:
print(f"\nStart date: {df['Date'].min()}")
print(f"End date: {df['Date'].max()}")
print(f"Total trading days: {len(df)}")


Start date: 1999-01-22
End date: 2025-02-14
Total trading days: 6558


In [10]:
date_range = pd.bdate_range(start=df['Date'].min(), end=df['Date'].max())
missing_dates = date_range.difference(df['Date'])
print(f"Missing business days (possible gaps): {len(missing_dates)}")

Missing business days (possible gaps): 6801


### OUTLIERS


In [11]:

print("\n── Outliers by column (IQR method) ──")
num_cols = ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"  {col}: {len(outliers)} outliers")



── Outliers by column (IQR method) ──
  Adj Close: 1139 outliers
  Close: 1138 outliers
  High: 1136 outliers
  Low: 1134 outliers
  Open: 1136 outliers
  Volume: 336 outliers


In [12]:
#Reduce data set to 2020-2024 for better model performance and relevance
df = df[df['Date'] >= '2020-01-01'].reset_index(drop=True)
df['Date'] = pd.to_datetime(df['Date'])
print(f"Data: {len(df)} | {df['Date'].min().date()} → {df['Date'].max().date()}")

Data: 1288 | 2020-01-02 → 2025-02-14


### Features for model

In [13]:

def add_features(df):
    d = df.copy()

    # Returns
    d['ret_1']  = d['Adj Close'].pct_change(1)
    d['ret_3']  = d['Adj Close'].pct_change(3)
    d['ret_5']  = d['Adj Close'].pct_change(5)
    d['ret_10'] = d['Adj Close'].pct_change(10)
    d['ret_20'] = d['Adj Close'].pct_change(20)

    # MAs relative to price
    for w in [5, 10, 20, 50]:
        d[f'ma{w}_ratio'] = d['Adj Close'].rolling(w).mean() / d['Adj Close'] - 1

    # Volatibility (std de retornos)
    d['vol_5']  = d['ret_1'].rolling(5).std()
    d['vol_10'] = d['ret_1'].rolling(10).std()
    d['vol_20'] = d['ret_1'].rolling(20).std()

    # RSI
    delta = d['Adj Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-8)))
    d['rsi'] = d['rsi'] / 100  # normalizar 0-1

    # MACD
    ema12 = d['Adj Close'].ewm(span=12).mean()
    ema26 = d['Adj Close'].ewm(span=26).mean()
    macd  = ema12 - ema26
    d['macd_norm']   = macd / d['Adj Close']
    d['macd_signal'] = macd.ewm(span=9).mean() / d['Adj Close']

    # Bollinger
    bb_mid = d['Adj Close'].rolling(20).mean()
    bb_std = d['Adj Close'].rolling(20).std()
    d['bb_pos'] = (d['Adj Close'] - bb_mid) / (2 * bb_std + 1e-8)  # -1 a +1

    # Candle
    d['hl_pct']   = (d['High'] - d['Low']) / d['Adj Close']
    d['oc_pct']   = (d['Adj Close'] - d['Open']) / d['Open']

    # Volume
    d['vol_ratio'] = np.log1p(d['Volume']) / np.log1p(d['Volume'].rolling(20).mean() + 1e-8) - 1

    return d.dropna().reset_index(drop=True)

df = add_features(df)
print(f"Post feature engineering: {len(df)} data")

Post feature engineering: 1239 data


### Normalization

In [14]:
FEATURE_COLS = [
    'ret_1', 'ret_3', 'ret_5', 'ret_10', 'ret_20',
    'ma5_ratio', 'ma10_ratio', 'ma20_ratio', 'ma50_ratio',
    'vol_5', 'vol_10', 'vol_20',
    'rsi', 'macd_norm', 'macd_signal', 'bb_pos',
    'hl_pct', 'oc_pct', 'vol_ratio'
]
N_FEATURES  = len(FEATURE_COLS)
WINDOW_SIZE = 30

In [15]:
scaler = MinMaxScaler(feature_range=(-1, 1))
features_scaled = scaler.fit_transform(df[FEATURE_COLS])

prices = df['Adj Close'].values

def create_sequences(features, prices, window):

    Xs, ys, price_refs = [], [], []
    for i in range(window, len(features)):
        Xs.append(features[i-window:i])

        ref_price = prices[i - window]
        ys.append((prices[i] / ref_price) - 1)
        price_refs.append(ref_price)
    return np.array(Xs), np.array(ys), np.array(price_refs)

X_all, y_all, refs_all = create_sequences(features_scaled, prices, WINDOW_SIZE)
print(f"Sequences: X={X_all.shape}, y={y_all.shape}")

Sequences: X=(1209, 30, 19), y=(1209,)


### Division Dataset

In [16]:
n         = len(X_all)
train_end = int(n * 0.72)
val_end   = int(n * 0.86)

X_train, y_train = X_all[:train_end],        y_all[:train_end]
X_val,   y_val   = X_all[train_end:val_end],  y_all[train_end:val_end]
X_test,  y_test  = X_all[val_end:],           y_all[val_end:]
refs_test        = refs_all[val_end:]

print(f"Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

Train=870 | Val=169 | Test=170


### Model

In [17]:
def build_model(window, n_features):
    inputs = Input(shape=(window, n_features))

    # CNN branch — short-term patterns
    c = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    c = Conv1D(32, kernel_size=3, activation='relu', padding='same')(c)
    c = GlobalAveragePooling1D()(c)

    # LSTM branch — long-term dependencies
    x = Bidirectional(LSTM(96, return_sequences=True))(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)
    x = Bidirectional(LSTM(48, return_sequences=False))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.20)(x)

    # Fusion
    merged = Concatenate()([x, c])
    out = Dense(64, activation='relu')(merged)
    out = Dropout(0.15)(out)
    out = Dense(32, activation='relu')(out)
    out = Dense(1)(out)

    return Model(inputs, out)

model = build_model(WINDOW_SIZE, N_FEATURES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss='huber',
    metrics=['mae']
)
model.summary()




Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 30, 19)]             0         []                            
                                                                                                  
 bidirectional (Bidirection  (None, 30, 192)              89088     ['input_1[0][0]']             
 al)                                                                                              
                                                                                                  
 layer_normalization (Layer  (None, 30, 192)              384       ['bidirectional[0][0]']       
 Normalization)                                                                                   
                                                                                            

### Training

In [18]:

callbacks = [
    EarlyStopping(patience=20, restore_best_weights=True,
                  monitor='val_loss', verbose=1),
    ReduceLROnPlateau(factor=0.4, patience=8,
                      min_lr=5e-7, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/200


28/28 [==============================] - 7s 70ms/step - loss: 0.0970 - mae: 0.3430 - val_loss: 0.0070 - val_mae: 0.0987 - lr: 2.0000e-04
Epoch 2/200
28/28 [==============================] - 1s 30ms/step - loss: 0.0430 - mae: 0.2314 - val_loss: 0.0081 - val_mae: 0.1096 - lr: 2.0000e-04
Epoch 3/200
28/28 [==============================] - 1s 28ms/step - loss: 0.0300 - mae: 0.1891 - val_loss: 0.0073 - val_mae: 0.1005 - lr: 2.0000e-04
Epoch 4/200
28/28 [==============================] - 1s 27ms/step - loss: 0.0236 - mae: 0.1713 - val_loss: 0.0087 - val_mae: 0.1061 - lr: 2.0000e-04
Epoch 5/200
28/28 [==============================] - 1s 31ms/step - loss: 0.0192 - mae: 0.1573 - val_loss: 0.0068 - val_mae: 0.0932 - lr: 2.0000e-04
Epoch 6/200
28/28 [==============================] - 1s 30ms/step - loss: 0.0153 - mae: 0.1372 - val_loss: 0.0065 - val_mae: 0.0943 - lr: 2.0000e-04
Epoch 7/200
28/28 [==============================] - 1s 30ms/step - loss: 0.0126 - mae: 0.1232 - val_los

### Evaluation

In [23]:

y_pred_norm = model.predict(X_test, verbose=0).flatten()
y_real_norm = y_test.flatten()

prices_pred = refs_test * (1 + y_pred_norm)
prices_real = refs_test * (1 + y_real_norm)

# Compparison against real prices from the dataframe (not just reconstructed from refs)
test_start_idx  = val_end + WINDOW_SIZE
prices_real_df  = df['Adj Close'].iloc[test_start_idx:test_start_idx + len(y_pred_norm)].values

mae  = mean_absolute_error(prices_real_df, prices_pred)
rmse = np.sqrt(mean_squared_error(prices_real_df, prices_pred))
r2   = r2_score(prices_real_df, prices_pred)
mape = np.mean(np.abs((prices_real_df - prices_pred) / (prices_real_df + 1e-8))) * 100

# Directional accuracy: % The percentage of times the model correctly predicts the direction of price movement (up/down)
dir_acc = np.mean(np.sign(y_pred_norm) == np.sign(y_real_norm)) * 100

print(" TEST SET")
print(f"  MAE  : ${mae:.4f}")
print(f"  RMSE : ${rmse:.4f}")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")
print(f"  Directional accuracy: {dir_acc:.1f}%")

test_dates = df['Date'].iloc[test_start_idx:test_start_idx + len(y_pred_norm)].values

 TEST SET
  MAE  : $7.1022
  RMSE : $9.0959
  R²   : 0.3986
  MAPE : 5.67%
  Directional accuracy: 84.1%


### Forcasting 30 days

In [20]:

last_features = features_scaled[-WINDOW_SIZE:]  # última ventana real
forecast_prices = []
forecast_dates  = pd.bdate_range(
    start=df['Date'].max() + pd.Timedelta(days=1), periods=30
)

current_window = last_features.copy()
last_price     = df['Adj Close'].iloc[-1]

for step in range(30):
    ref_price = prices[-(WINDOW_SIZE - step)] if step < WINDOW_SIZE else last_price

    pred_norm = model.predict(
        current_window[np.newaxis, :, :], verbose=0
    )[0, 0]

    new_price = ref_price * (1 + pred_norm)
    forecast_prices.append(new_price)

    new_ret    = pred_norm 
    new_row    = current_window[-1].copy()
    new_row[0] = np.clip(new_ret, -0.5, 0.5)  # ret_1 aprox
    current_window = np.vstack([current_window[1:], new_row])

forecast_prices = np.array(forecast_prices)

In [21]:


# ── VISUALIZACIÓN CON PLOTLY ──────────────────────────────────

fig = make_subplots(
    rows=3, cols=2,
    specs=[[{"colspan": 2}, None],
           [{"colspan": 2}, None],
           [{"type": "scatter"}, {"type": "scatter"}]],
    subplot_titles=(
        'NVDA — Serie histórica (2020 → actualidad)',
        f'Test — Real vs Predicción  |  MAE=${mae:.2f}  RMSE=${rmse:.2f}  R²={r2:.3f}  Dir={dir_acc:.1f}%',
        'Forecast — Próximos 30 días hábiles',
        'Curva de Aprendizaje'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# ── Plot 1: Serie histórica ───────────────────────────────────
fig.add_trace(go.Scatter(
    x=df['Date'], y=df['Adj Close'],
    mode='lines',
    name='Precio real',
    line=dict(color='#00e5ff', width=1.5),
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Precio: $%{y:.2f}<extra></extra>'
), row=1, col=1)

# Líneas verticales Train|Val y Val|Test
train_date = df['Date'].iloc[train_end + WINDOW_SIZE]
val_date   = df['Date'].iloc[val_end   + WINDOW_SIZE]
ymax = df['Adj Close'].max()

for xval, label, color in [
    (train_date, 'Train | Val', 'rgba(255,255,255,0.5)'),
    (val_date,   'Val | Test',  'rgba(255,214,10,0.7)')
]:
    fig.add_vline(x=xval, line=dict(color=color, dash='dash', width=1.5), row=1, col=1)
    fig.add_annotation(x=xval, y=ymax * 0.9, text=label,
                       font=dict(color=color, size=10),
                       showarrow=False, xanchor='left', row=1, col=1)

# ── Plot 2: Real vs Predicción ────────────────────────────────
fig.add_trace(go.Scatter(
    x=test_dates, y=prices_real_df,
    mode='lines', name='Real',
    line=dict(color='#00e5ff', width=2),
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Real: $%{y:.4f}<extra></extra>'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=test_dates, y=prices_pred,
    mode='lines', name='Predicción',
    line=dict(color='#ff6b35', width=2, dash='dash'),
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Pred: $%{y:.4f}<extra></extra>'
), row=2, col=1)

# Área de error entre real y pred
fig.add_trace(go.Scatter(
    x=list(test_dates) + list(test_dates[::-1]),
    y=list(prices_real_df) + list(prices_pred[::-1]),
    fill='toself',
    fillcolor='rgba(255,107,53,0.08)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False, hoverinfo='skip'
), row=2, col=1)

# ── Plot 3: Forecast ──────────────────────────────────────────
recent = df.tail(90)
fig.add_trace(go.Scatter(
    x=recent['Date'], y=recent['Adj Close'],
    mode='lines', name='Histórico reciente',
    line=dict(color='#00e5ff', width=1.8),
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Precio: $%{y:.2f}<extra></extra>'
), row=3, col=1)

# Calcular % cambio para cada punto del forecast
last_price = df['Adj Close'].iloc[-1]
forecast_hover = [
    f'<b>{d.date()}</b><br>Forecast: ${p:.4f}<br>Δ vs hoy: {((p-last_price)/last_price*100):+.2f}%<extra></extra>'
    for d, p in zip(forecast_dates, forecast_prices)
]

fig.add_trace(go.Scatter(
    x=forecast_dates, y=forecast_prices,
    mode='lines+markers',
    name='Forecast 30d',
    line=dict(color='#a8ff78', width=2),
    marker=dict(size=4, color='#a8ff78'),
    hovertemplate=forecast_hover
), row=3, col=1)

fig.add_vline(x=df['Date'].max(),
              line=dict(color='rgba(255,255,255,0.4)', dash='dash', width=1),
              row=3, col=1)

# ── Plot 4: Loss curve ────────────────────────────────────────
epochs = list(range(1, len(history.history['loss']) + 1))

fig.add_trace(go.Scatter(
    x=epochs, y=history.history['loss'],
    mode='lines', name='Train Loss',
    line=dict(color='#00e5ff', width=2),
    hovertemplate='Época %{x}<br>Train Loss: %{y:.6f}<extra></extra>'
), row=3, col=2)

fig.add_trace(go.Scatter(
    x=epochs, y=history.history['val_loss'],
    mode='lines', name='Val Loss',
    line=dict(color='#ff4d6d', width=2, dash='dash'),
    hovertemplate='Época %{x}<br>Val Loss: %{y:.6f}<extra></extra>'
), row=3, col=2)

# ── Estilo global ─────────────────────────────────────────────
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0f0f1a',
    plot_bgcolor='#1a1a2e',
    title=dict(
        text='NVDA Stock — CNN + Bidirectional LSTM (v3)',
        font=dict(size=20, color='white', family='Arial Black'),
        x=0.5, xanchor='center'
    ),
    font=dict(color='white', family='Arial'),
    height=1100,
    legend=dict(
        bgcolor='rgba(26,26,46,0.8)',
        bordercolor='#444',
        borderwidth=1
    ),
    hovermode='x unified'  # hover sincronizado en todos los plots
)

# Ejes con estilo uniforme
fig.update_xaxes(gridcolor='rgba(255,255,255,0.08)', linecolor='#444', tickfont=dict(color='white'))
fig.update_yaxes(gridcolor='rgba(255,255,255,0.08)', linecolor='#444', tickfont=dict(color='white'),
                 tickprefix='$')

fig.show(renderer="browser")

In [24]:

for d, p in zip(forecast_dates[:10], forecast_prices[:10]):
    delta_pct = (p - df['Adj Close'].iloc[-1]) / df['Adj Close'].iloc[-1] * 100
    sign = "▲" if p > df['Adj Close'].iloc[-1] else "▼"
    print(f"  {d.date()}  →  ${p:.4f}  {sign} {delta_pct:+.2f}% vs today")
print("  ...")
print(f"  {forecast_dates[-1].date()}  →  ${forecast_prices[-1]:.4f}")
delta = forecast_prices[-1] - df['Adj Close'].iloc[-1]
print(f"\nActual Price:   ${df['Adj Close'].iloc[-1]:.4f}")
print(f"Price day+30:   ${forecast_prices[-1]:.4f}")
print(f"Estimated Change: {'+' if delta >= 0 else ''}{delta:.4f} USD ({delta/df['Adj Close'].iloc[-1]*100:+.2f}%)")
print(f"Correct Direction in Test: {dir_acc:.1f}%")

  2025-02-17  →  $135.5255  ▼ -2.39% vs today
  2025-02-18  →  $144.9337  ▲ +4.38% vs today
  2025-02-19  →  $151.5596  ▲ +9.15% vs today
  2025-02-20  →  $143.3560  ▲ +3.25% vs today
  2025-02-21  →  $143.4947  ▲ +3.35% vs today
  2025-02-24  →  $140.0096  ▲ +0.84% vs today
  2025-02-25  →  $138.1108  ▼ -0.53% vs today
  2025-02-26  →  $137.1055  ▼ -1.26% vs today
  2025-02-27  →  $141.8213  ▲ +2.14% vs today
  2025-02-28  →  $139.5656  ▲ +0.52% vs today
  ...
  2025-03-28  →  $165.6820

Actual Price:   $138.8500
Price day+30:   $165.6820
Estimated Change: +26.8320 USD (+19.32%)
Correct Direction in Test: 84.1%
